# Hexene epoxidation — NEB coadsorption model

Keep original plotting/export workflows. RP is a **provisional** effective TS. CatMAP linear scaling may alter effective barriers; inspect `ts_scaling_audit.csv` before interpreting TOF.


In [ ]:
from pathlib import Path
import os
from catmap import ReactionModel

project = Path.cwd()
if not (project / 'alkene_epoxidation.mkm').is_file():
    candidates = [project / 'lyq_alkene_epoxidation1' / '2-creating_microkinetic_model',
                  project / '2-creating_microkinetic_model']
    matches = [p for p in candidates if (p / 'alkene_epoxidation.mkm').is_file()]
    if not matches:
        raise FileNotFoundError('Open the new model directory. Current directory: ' + str(project))
    project = matches[0]
os.chdir(project)

model = ReactionModel(setup_file='alkene_epoxidation.mkm')
for variable in ['production_rate', 'rate', 'coverage']:
    if variable not in model.output_variables:
        model.output_variables.append(variable)

# CatMAP text-header serialization cannot eval certain numpy/mpmath repr values.
# Its built-in _pickle_attrs stores them in .pkl instead, without changing energetics.
for attr in ['adsorbate_coefficient_matrix', 'coefficient_matrix', 'kBT',
             'total_coefficient_dict', 'transition_state_coefficient_matrix',
             'transition_state_scaling_matrix', 'coverage_map', 'rate_map',
             'production_rate_map']:
    if attr not in model._pickle_attrs:
        model._pickle_attrs.append(attr)

# Ignore old map caches which were saved with only ['coverage', 'rate'].
model.force_recalculation = True
model.run()
print('Model directory:', Path.cwd())
print('Output variables:', model.output_variables)


In [ ]:
import math

resolution = model.resolution
if isinstance(resolution, (list, tuple)):
    expected_points = math.prod(int(v) for v in resolution)
else:
    expected_points = int(resolution) ** len(model.descriptor_names)

print('Expected descriptor grid points:', expected_points)
for name in ['production_rate', 'rate', 'coverage']:
    data = getattr(model, name + '_map', None)
    actual = len(data) if data is not None else 0
    print(name, 'points:', actual)
    if actual != expected_points:
        raise RuntimeError(f'{name}: expected {expected_points}, got {actual}')

print('Coverage labels:', model.output_labels['coverage'])
bad = []
for point, coverages in model.coverage_map:
    vals = [float(v) for v in coverages]
    if (not all(math.isfinite(v) for v in vals) or
            min(vals) < -1e-8 or max(vals) > 1+1e-8 or sum(vals) > 1+1e-6):
        bad.append((point, vals))
print('Coverage physical-range warnings:', len(bad))
for item in bad[:5]:
    print('WARNING:', item)


In [ ]:
import csv
from pathlib import Path

with Path('energies.txt').open(encoding='utf-8', newline='') as fh:
    source_rows = list(csv.DictReader(fh, delimiter='\t'))
by_surface = {}
for row in source_rows:
    if row['surface_name'] != 'None':
        by_surface.setdefault(row['surface_name'], {})[row['species_name']] = float(row['formation_energy'])

records = []
for surface in model.surface_names:
    inp = by_surface[surface]
    scaled = model.scaler.get_electronic_energies([inp['R'], inp['P']])
    actual_RP = float(scaled['RP_s'])
    barrier_actual = max(0., actual_RP-inp['R'], inp['P']-inp['R'])
    records.append({
        'surface': surface, 'R_input': inp['R'], 'P_input': inp['P'],
        'RP_input': inp['RP'], 'RP_scaled': actual_RP,
        'RP_scaling_error_eV': actual_RP-inp['RP'],
        'barrier_input_eV': max(0., inp['RP']-inp['R']),
        'barrier_scaled_eV': barrier_actual,
        'R_scaling_error_eV': float(scaled['R_s'])-inp['R'],
        'P_scaling_error_eV': float(scaled['P_s'])-inp['P']})

with Path('ts_scaling_audit.csv').open('w',encoding='utf-8-sig',newline='') as fh:
    writer = csv.DictWriter(fh,fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
print('Saved:', Path('ts_scaling_audit.csv').resolve())
for row in sorted(records,key=lambda r:abs(r['RP_scaling_error_eV']),reverse=True)[:8]:
    print(f"{row['surface']}: input Ea={row['barrier_input_eV']:.3f}, "
          f"scaled Ea={row['barrier_scaled_eV']:.3f}, "
          f"TS error={row['RP_scaling_error_eV']:+.3f} eV")
if any(abs(r['RP_scaling_error_eV']) > 0.05 for r in records):
    print('WARNING: RP is changed by linear fitting; plotted TOF is NOT exact per-surface NEB kinetics.')


In [ ]:
from pathlib import Path
from datetime import datetime
from catmap import analyze
import matplotlib.pyplot as plt

# Windows may lock PDFs that are open in a viewer. Keep the standard output
# names when possible; fall back to an independent folder on PermissionError.
backup_dir = Path("plots") / ("rerun_" + datetime.now().strftime("%Y%m%d_%H%M%S"))

def render_with_fallback(plotter, filename):
    try:
        fig = plotter.plot(save=filename)
        saved = Path(filename)
    except PermissionError:
        backup_dir.mkdir(parents=True, exist_ok=True)
        saved = backup_dir / filename
        print(f"WARNING: {filename} is locked or not writable; trying {saved}")
        fig = plotter.plot(save=str(saved))
    if fig is not None:
        plt.close(fig)
    print("Saved", saved.resolve())
    return saved

def plot_vector(variable, filename, log=False, labels=None,
                unique_only=True, minimum=None, maximum=None):
    if not getattr(model, variable + '_map', None):
        raise RuntimeError(f"{variable}_map missing: compute it in the first cell")
    vm = analyze.VectorMap(model)
    vm.plot_variable = variable
    vm.log_scale = log
    vm.unique_only = unique_only
    vm.include_labels = labels
    vm.descriptor_labels = [
        "R formation free energy [eV]", "P formation free energy [eV]"
    ]
    vm.subplots_adjust_kwargs = {"left": 0.2, "right": 0.8, "bottom": 0.15}
    if minimum is not None:
        vm.min = minimum
    if maximum is not None:
        vm.max = maximum
    if log:
        vm.threshold = 1e-30
    return render_with_fallback(vm, filename)

# Net elementary rates are signed. Plot them on a LINEAR scale to retain signs
# and avoid invalid log10 of negative reverse rates.
plot_vector("rate", "rate.pdf")
plot_vector("rate", "all_rates.pdf", unique_only=False)

# Plot only epoxide production on logarithmic scale. This series is nonnegative
# in the last saved run; recomputed results are explicitly checked below.
product_label = "C6H12O_g"
prod_labels = list(model.output_labels["production_rate"])
if product_label not in prod_labels:
    raise ValueError(f"No {product_label} in production-rate output labels {prod_labels}")
product_index = prod_labels.index(product_label)
product_values = [
    float(values[product_index]) for _, values in model.production_rate_map
]
if any(v < 0 for v in product_values):
    print("WARNING: negative epoxide net production encountered; using linear scale.")
    product_log = False
else:
    product_log = True
plot_vector("production_rate", "production_rate.pdf", log=product_log,
            labels=[product_label],
            minimum=1e-30 if product_log else None)
plot_vector("production_rate", "pretty_production_rate.pdf", log=product_log,
            labels=[product_label],
            minimum=1e-30 if product_log else None)

plot_vector("coverage", "coverage.pdf", minimum=0, maximum=1)
plot_vector("coverage", "R_P_coverage.pdf",
            labels=["R_s", "P_s"], minimum=0, maximum=1)


In [ ]:
from pathlib import Path
from catmap import analyze
import matplotlib.pyplot as plt

# Reuse the robust PDF saver created in the preceding plotting cell.
sa = analyze.ScalingAnalysis(model)
render_with_fallback(sa, "scaling.pdf")


In [ ]:
import csv
from pathlib import Path

variable = 'production_rate'
labels = [str(v) for v in model.output_labels[variable]]
out = Path(variable+'_table.txt')
with out.open('w',encoding='utf-8',newline='') as fh:
    writer = csv.writer(fh,delimiter='\t')
    writer.writerow(['descriptor-'+d for d in model.descriptor_names] + labels)
    for point, rates in getattr(model,variable+'_map'):
        writer.writerow([float(x) for x in point]+[float(x) for x in rates])
print('Wrote',out.resolve())


In [ ]:
point, cvg = model.coverage_map[0]
print('descriptors:',point)
print('species:',model.output_labels['coverage'])
print('coverages:',[float(v) for v in cvg])


In [ ]:
# Optional expensive sensitivity calculation; run only if explicitly needed.
# for name in ['rate_control', 'selectivity_control', 'rxn_order']:
#     if name not in model.output_variables:
#         model.output_variables.append(name)
# model.run()
# mm = analyze.MatrixMap(model)
# mm.plot_variable = 'rate_control'
# mm.log_scale = False
# mm.min, mm.max = -2, 2
# mm.plot(save='rate_control.pdf')
